In [1]:
from qat.external.utils.qroutines.hamming_weight_compute import fpc
from qat.external.utils.qroutines import qregs_init as qregs
from qat.lang.AQASM import *
from qat.qpus import PyLinalg
from math import factorial

In [2]:
qpu = PyLinalg()

# Compute

In [3]:
def hamming_weight_compute(bitstring):
    # this function only gets useful patterns to later generate the circuit
    nwr_dict = fpc.get_qroutine_for_qubits_weight_get_pattern(len(bitstring))
    print("nwr_dict ", nwr_dict)
    program = Program()

    a = program.qalloc(nwr_dict['n_lines'])
    cout = program.qalloc(nwr_dict['n_couts'])

    # initialize a set of qubits starting from bitstring
    # basically, it applies X gate to qubit when corresponding index in bitstring is 1
    qfun = qregs.initialize_qureg_given_bitstring(bitstring, True)
    program.apply(qfun, a)

    qfun = fpc.get_qroutine_for_qubits_weight(len(a), len(cout), nwr_dict)
    program.apply(qfun, a, cout)

    to_measure_qubits = fpc.get_to_measure_qubits(a, cout, nwr_dict)

    res = qpu.submit(program.to_circ().to_job(
        qubits=[qb.index for qb in to_measure_qubits]))
    print("res ", res)

    counts = len(res.raw_data)
    assert counts == 1
    exp_w = bitstring.count("1") # count the no. of ones
    state = res.raw_data[0].state
    assert state.lsb_int == exp_w

In [4]:
for bitstring in [
        ("0000"),
        ("0101"),
        ("0001"),
        ("1101"),
        ("1001"),
        ("1111"),
        ("10110100"),
        ("11001011"),
        ("11010000"),
]:
    print(bitstring)
    hamming_weight_compute(bitstring)
    print("-"*20)

0000
nwr_dict  {'n_lines': 4, 'n_couts': 3, 'adders_pattern': [('a0', 'a1', 'c0'), ('a2', 'a3', 'c1'), ('a1', 'c0', 'a3', 'c1', 'c2')], 'results': ['a3', 'c1', 'c2']}
res  Result(need_flip=False, lsb_first=False, nbqbits=None, has_statevector=False, statevector=None, data=None, _value=None, raw_data=[Sample(_amplitude=None, probability=1.0, _state=0, err=None, intermediate_measurements=[], qregs=[DefaultRegister(key=None, length=3, start=0, msb=None)])], qregs=[DefaultRegister(key=None, length=3, start=0, msb=None)], error=None, value_data=None, error_data=None, meta_data={}, in_memory=None)
--------------------
0101
nwr_dict  {'n_lines': 4, 'n_couts': 3, 'adders_pattern': [('a0', 'a1', 'c0'), ('a2', 'a3', 'c1'), ('a1', 'c0', 'a3', 'c1', 'c2')], 'results': ['a3', 'c1', 'c2']}
res  Result(need_flip=False, lsb_first=False, nbqbits=None, has_statevector=False, statevector=None, data=None, _value=None, raw_data=[Sample(_amplitude=None, probability=1.0, _state=2, err=None, intermediate_meas

# Check

In [17]:
def hamming_weight_check(weight_int, n_bits):
    nwr_dict = fpc.get_qroutine_for_qubits_weight_get_pattern(n_bits)

    program = Program()
    a = program.qalloc(nwr_dict['n_lines'])
    cout = program.qalloc(nwr_dict['n_couts'])
    eq = program.qalloc(1)
    # create a superposition of inputs
    for qb in a:
        program.apply(H, qb)
        
    # ... and check the weight for all
    qfun = fpc.get_qroutine_for_qubits_weight_check(
        len(a), len(cout), weight_int, nwr_dict, True)
    program.apply(qfun, a, cout, eq)
    qfun = fpc.get_qroutine_for_qubits_weight_check(len(a),
                                                    len(cout),
                                                    weight_int,
                                                    nwr_dict,
                                                    False)
    ### The idea is to do an uncompute in order to leave only the 
    # eq qubit set to 1 (if the hamming weight check is positive), 
    # while restoring all other ones
    program.apply(qfun.dag(), a, cout)
    res = qpu.submit(program.to_circ().to_job(qubits=[a, eq]))

    counts = len(res.raw_data)
    assert counts == 2**len(a)
    total_actives = 0
    for sample in res:
        state = sample.state
        bitstring = state.bitstring
        print(f"{state} -> {bitstring}")
        if bitstring[-1] == '1':
            total_actives += 1
            # + 1 bcz we have the eq qbits, should be faster than slicing
            assert bitstring.count("1") == weight_int + 1
        else:
            assert bitstring.count("1") !=  weight_int, f"{bitstring} and {weight_int}"
            
    # n choose r, computed directly
    exp_actives = factorial(n_bits) / factorial(weight_int) / factorial(
        n_bits - weight_int)
    assert total_actives == exp_actives, f"{total_actives}, {exp_actives}"

In [20]:
for name, weight_int, n_bits in [
    ("0on2", 0, 2),
    ("1on2", 1, 2),
    ("2on2", 2, 2),
    ("0on4", 0, 4),
    ("2on4", 2, 4),
    ("3on4", 3, 4),
    ("3on8", 1, 8),
    ("3on8", 2, 8),
    ("3on8", 3, 8),
    ("3on8", 4, 8),
                                ]:
    print(f"Set eq qubit to 1 iff the weight of {name[-1]} qubit is equal to {name[0]}")
    hamming_weight_check(weight_int, n_bits)

Set eq qubit to 1 iff the weight of 2 qubit is equal to 0
|00>|1> -> 001
|01>|0> -> 010
|10>|0> -> 100
|11>|0> -> 110
Set eq qubit to 1 iff the weight of 2 qubit is equal to 1
|00>|0> -> 000
|01>|1> -> 011
|10>|1> -> 101
|11>|0> -> 110
Set eq qubit to 1 iff the weight of 2 qubit is equal to 2
|00>|0> -> 000
|01>|0> -> 010
|10>|0> -> 100
|11>|1> -> 111
Set eq qubit to 1 iff the weight of 4 qubit is equal to 0
|0000>|1> -> 00001
|0001>|0> -> 00010
|0010>|0> -> 00100
|0011>|0> -> 00110
|0100>|0> -> 01000
|0101>|0> -> 01010
|0110>|0> -> 01100
|0111>|0> -> 01110
|1000>|0> -> 10000
|1001>|0> -> 10010
|1010>|0> -> 10100
|1011>|0> -> 10110
|1100>|0> -> 11000
|1101>|0> -> 11010
|1110>|0> -> 11100
|1111>|0> -> 11110
Set eq qubit to 1 iff the weight of 4 qubit is equal to 2
|0000>|0> -> 00000
|0001>|0> -> 00010
|0010>|0> -> 00100
|0011>|1> -> 00111
|0100>|0> -> 01000
|0101>|1> -> 01011
|0110>|1> -> 01101
|0111>|0> -> 01110
|1000>|0> -> 10000
|1001>|1> -> 10011
|1010>|1> -> 10101
|1011>|0> -> 1011

|00000000>|0> -> 000000000
|00000001>|0> -> 000000010
|00000010>|0> -> 000000100
|00000011>|1> -> 000000111
|00000100>|0> -> 000001000
|00000101>|1> -> 000001011
|00000110>|1> -> 000001101
|00000111>|0> -> 000001110
|00001000>|0> -> 000010000
|00001001>|1> -> 000010011
|00001010>|1> -> 000010101
|00001011>|0> -> 000010110
|00001100>|1> -> 000011001
|00001101>|0> -> 000011010
|00001110>|0> -> 000011100
|00001111>|0> -> 000011110
|00010000>|0> -> 000100000
|00010001>|1> -> 000100011
|00010010>|1> -> 000100101
|00010011>|0> -> 000100110
|00010100>|1> -> 000101001
|00010101>|0> -> 000101010
|00010110>|0> -> 000101100
|00010111>|0> -> 000101110
|00011000>|1> -> 000110001
|00011001>|0> -> 000110010
|00011010>|0> -> 000110100
|00011011>|0> -> 000110110
|00011100>|0> -> 000111000
|00011101>|0> -> 000111010
|00011110>|0> -> 000111100
|00011111>|0> -> 000111110
|00100000>|0> -> 001000000
|00100001>|1> -> 001000011
|00100010>|1> -> 001000101
|00100011>|0> -> 001000110
|00100100>|1> -> 001001001
|

|00000000>|0> -> 000000000
|00000001>|0> -> 000000010
|00000010>|0> -> 000000100
|00000011>|0> -> 000000110
|00000100>|0> -> 000001000
|00000101>|0> -> 000001010
|00000110>|0> -> 000001100
|00000111>|0> -> 000001110
|00001000>|0> -> 000010000
|00001001>|0> -> 000010010
|00001010>|0> -> 000010100
|00001011>|0> -> 000010110
|00001100>|0> -> 000011000
|00001101>|0> -> 000011010
|00001110>|0> -> 000011100
|00001111>|1> -> 000011111
|00010000>|0> -> 000100000
|00010001>|0> -> 000100010
|00010010>|0> -> 000100100
|00010011>|0> -> 000100110
|00010100>|0> -> 000101000
|00010101>|0> -> 000101010
|00010110>|0> -> 000101100
|00010111>|1> -> 000101111
|00011000>|0> -> 000110000
|00011001>|0> -> 000110010
|00011010>|0> -> 000110100
|00011011>|1> -> 000110111
|00011100>|0> -> 000111000
|00011101>|1> -> 000111011
|00011110>|1> -> 000111101
|00011111>|0> -> 000111110
|00100000>|0> -> 001000000
|00100001>|0> -> 001000010
|00100010>|0> -> 001000100
|00100011>|0> -> 001000110
|00100100>|0> -> 001001000
|